In [1]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU device: {torch.cuda.get_device_name(0)}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

CUDA available: True
CUDA version: 12.1
PyTorch version: 2.5.1+cu121
GPU device: NVIDIA GeForce RTX 4060 Laptop GPU
Number of GPUs: 1


In [1]:
import logging

from dotenv import load_dotenv
from langchain_core.documents import Document

from rag.ingestion.llama_parse_processor import process_document
from rag.ingestion.document_fetcher import DocumentFetcher
from rag.ingestion.document_tracker import DocumentTracker
from rag.ingestion.vector_store import QdrantManager
from utils.api_utils import DefineEdgeFundamentalsAPI
from utils.data_helpers import (
    initialize_metadata_data,
    initialize_stock_data,
    symbol_to_fincode,
)

load_dotenv()

logger = logging.getLogger(__name__)


await initialize_stock_data()
await initialize_metadata_data()

c:\Users\Anant\Downloads\embed-documents-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-08 21:43:29,638 - utils.data_helpers - INFO - Initializing stock data mappings...
2026-02-08 21:43:29,654 - utils.api_utils - INFO - Retrieved active equity and sub listing data from cache
2026-02-08 21:43:29,676 - utils.data_helpers - INFO - Stock data initialized: 5517 stocks, 5501 symbols
2026-02-08 21:43:29,677 - utils.data_helpers - INFO - Initializing metadata mappings (20 years of data)...
2026-02-08 21:43:29,697 - utils.api_utils - INFO - Retrieved meta data file for 2006-02-13 to 2026-02-08 from cache
2026-02-08 21:43:29,717 - utils.data_helpers - INFO - Metadata initialized: 22109 items, 4064 unique fincodes


### Get Stocks from Nifty 50 Index

In [2]:
api = DefineEdgeFundamentalsAPI()

groups = await api.get_predefined_groups()

all_group_names = groups.keys()
for gname in all_group_names:
    if gname == "Nifty 50 Index":
        nifty_50_stocks = groups[gname]

symbol_to_fincode_map = {}
for symbol in nifty_50_stocks:
    symbol_to_fincode_map[symbol] = symbol_to_fincode(symbol)

fincodes = list(symbol_to_fincode_map.values())

2026-02-08 21:43:39,478 - httpx - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/groups/0 "HTTP/1.1 200 OK"


In [3]:
import os

fetcher = DocumentFetcher()
doc_tracker = DocumentTracker()
qdrant_manager = QdrantManager() 

# Calculate optimal number of workers (75% of CPU cores)
max_workers = max(1, int(os.cpu_count() * 0.75))
logger.info(f"Using {max_workers} parallel workers (75% of {os.cpu_count()} cores)")

2026-02-08 21:43:46,050 - rag.ingestion.document_tracker - INFO - Initialized DocumentTracker with database at c:\Users\Anant\Downloads\embed-documents-main\embed-documents-main\rag\data\embedded_docs.db
2026-02-08 21:43:47,446 - rag.ingestion.document_tracker - INFO - Initialized DocumentTracker with database at c:\Users\Anant\Downloads\embed-documents-main\embed-documents-main\rag\data\embedded_docs.db
2026-02-08 21:43:47,447 - rag.ingestion.vector_store - INFO - Initialized QdrantManager with collection 'company_files'
2026-02-08 21:43:47,448 - __main__ - INFO - Using 13 parallel workers (75% of 18 cores)


In [4]:
import asyncio

# Create a semaphore to limit concurrent processing
semaphore = asyncio.Semaphore(max_workers)

async def process_fincode(fincode: str, semaphore: asyncio.Semaphore):
    """Process a single fincode: fetch, check existence, process, and embed documents."""
    async with semaphore:
        logger.info(f"Starting processing for fincode: {fincode}")
        
        # fetch documents for the fincode
        try:
            docs = await fetcher.get_available_documents(
                fincode=fincode,
            )
            for doc in docs:
                logger.info(
                    f"Document: {doc.filename} ({doc.category}) - {doc.document_date}"
                )
        except Exception as e:
            logger.error(f"Error fetching documents for fincode {fincode}: {e}")
            return

        try:
            # check which documents already exist in vector store
            exists = await doc_tracker.check_documents_exist([doc.filename for doc in docs])
        except Exception as e:
            logger.error(f"Error checking document existence for fincode {fincode}: {e}")
            return

        # process fincode documents
        try:
            processed_docs: list[Document] = []

            for doc in docs:
                if not exists[doc.filename]:
                    try:
                        logger.info(
                            f"Document {doc.filename} does not exist in vector store. Proceeding to fetch and save."
                        )
                        pdf = await fetcher.get_pdf_doc_stream(doc.filename, doc.category)

                        file_size = len(pdf.stream.getbuffer()) / 1024 / 1024

                        logger.info(f"Document size: {file_size:,} bytes ({file_size:.2f} MB)")

                        processed_docs.extend(await process_document(pdf, file_type="pdf"))
                    except Exception as e:
                        logger.error(f"Error processing document {doc.filename}: {e}")
                        continue
        except Exception as e:
            logger.error(f"Error processing documents for fincode {fincode}: {e}")
            return

        # Select documents that need to be embedded
        try:
            docs_to_embed: list[Document] = []
            for doc in processed_docs:
                if not exists[doc.metadata.get("source", "")]:
                    docs_to_embed.append(doc)

            for doc in docs_to_embed:
                logger.info(f"Document to embed: {doc.metadata.get('source', '')}")
        except Exception as e:
            logger.error(f"Error selecting documents to embed for fincode {fincode}: {e}")
            if not docs_to_embed:
                logger.info(f"No new documents to embed for fincode {fincode}. Skipping.")
                return
        
        try:
            if isinstance(docs_to_embed, list) and len(docs_to_embed) > 0:
                result = await qdrant_manager.embed_documents(docs_to_embed)
                logger.info(f"Fincode {fincode} - Cost: ${result['estimated_cost_usd']}")
        except Exception as e:
            logger.error(f"Error embedding documents for fincode {fincode}: {e}")
            return
        
        logger.info(f"Completed processing for fincode: {fincode}")

In [5]:
# Process all fincodes in parallel
logger.info(f"Starting parallel processing of {len(fincodes)} fincodes...")
tasks = [process_fincode(fincode, semaphore) for fincode in fincodes]
await asyncio.gather(*tasks)
logger.info("All fincodes processed!")

2026-02-08 21:43:59,846 - __main__ - INFO - Starting parallel processing of 50 fincodes...
2026-02-08 21:43:59,848 - __main__ - INFO - Starting processing for fincode: 112599
2026-02-08 21:43:59,849 - rag.ingestion.document_fetcher - INFO - Fetching document metadata: fincode=112599, category=None, date_range=2024-02-09 to 2026-02-08
2026-02-08 21:43:59,850 - __main__ - INFO - Starting processing for fincode: 132921
2026-02-08 21:43:59,851 - rag.ingestion.document_fetcher - INFO - Fetching document metadata: fincode=132921, category=None, date_range=2024-02-09 to 2026-02-08
2026-02-08 21:43:59,854 - __main__ - INFO - Starting processing for fincode: 108869
2026-02-08 21:43:59,859 - rag.ingestion.document_fetcher - INFO - Fetching document metadata: fincode=108869, category=None, date_range=2024-02-09 to 2026-02-08
2026-02-08 21:43:59,864 - __main__ - INFO - Starting processing for fincode: 100820
2026-02-08 21:43:59,867 - rag.ingestion.document_fetcher - INFO - Fetching document metada

Started parsing the file under job_id 9b3252b9-f3fd-4ec4-a6b9-4705d34c51e5
Started parsing the file under job_id 6aaae4bd-93ad-41da-8634-b26aaf38d53f


2026-02-08 21:44:12,561 - __main__ - INFO - Document size: 4.10960578918457 bytes (4.11 MB)
2026-02-08 21:44:12,999 - httpx - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 200 OK"


Started parsing the file under job_id 7e60e93b-335b-4c0d-ad7e-ffe1ff661494


2026-02-08 21:44:13,808 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/9b3252b9-f3fd-4ec4-a6b9-4705d34c51e5 "HTTP/1.1 200 OK"
2026-02-08 21:44:13,942 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/6aaae4bd-93ad-41da-8634-b26aaf38d53f "HTTP/1.1 200 OK"
2026-02-08 21:44:14,377 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7e60e93b-335b-4c0d-ad7e-ffe1ff661494 "HTTP/1.1 200 OK"
2026-02-08 21:44:16,175 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/9b3252b9-f3fd-4ec4-a6b9-4705d34c51e5 "HTTP/1.1 200 OK"
2026-02-08 21:44:16,313 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/6aaae4bd-93ad-41da-8634-b26aaf38d53f "HTTP/1.1 200 OK"
2026-02-08 21:44:16,811 - httpx - INFO - HTTP Request: GET https://api.cloud.llamaindex.ai/api/parsing/job/7e60e93b-335b-4c0d-ad7e-ffe1ff661494 "HTTP/1.1 200 OK"
2026-02-08 21:44:16,871 - ht

...

Error while parsing the file 'C:\Users\Anant\AppData\Local\Temp\tmp3d0_x5gs.pdf': Failed to parse the file: {"detail":"You've exceeded the maximum number of credits for your plan."}


2026-02-08 22:13:24,019 - httpx - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/getFileByCategoryAndAttachmentName?subcatname=concall&attachmentName=053a0123-0143-4181-b688-a0576aca84c1.pdf "HTTP/1.1 200 OK"
2026-02-08 22:13:24,275 - __main__ - INFO - Document size: 0.363494873046875 bytes (0.36 MB)
2026-02-08 22:13:28,864 - httpx - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 402 Payment Required"
2026-02-08 22:13:28,872 - __main__ - INFO - Document 25f69d4b-4b90-4b69-80cb-da8b38788d9d.pdf does not exist in vector store. Proceeding to fetch and save.


Error while parsing the file 'C:\Users\Anant\AppData\Local\Temp\tmp977kp2dm.pdf': Failed to parse the file: {"detail":"You've exceeded the maximum number of credits for your plan."}


2026-02-08 22:13:31,838 - httpx - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/getFileByCategoryAndAttachmentName?subcatname=investor-presentation&attachmentName=25f69d4b-4b90-4b69-80cb-da8b38788d9d.pdf "HTTP/1.1 200 OK"
2026-02-08 22:13:34,343 - __main__ - INFO - Document size: 10.519454002380371 bytes (10.52 MB)
2026-02-08 22:13:41,717 - httpx - INFO - HTTP Request: POST https://api.cloud.llamaindex.ai/api/parsing/upload "HTTP/1.1 402 Payment Required"
2026-02-08 22:13:41,729 - __main__ - INFO - Completed processing for fincode: 100114
2026-02-08 22:13:41,732 - __main__ - INFO - All fincodes processed!


Error while parsing the file 'C:\Users\Anant\AppData\Local\Temp\tmpv0fiz4va.pdf': Failed to parse the file: {"detail":"You've exceeded the maximum number of credits for your plan."}
